# 🔍 Topic 02: Convolution Operation & Filters from Scratch and Keras
### 🧮 Pure NumPy Implementation + Real Image Filtering + Keras Conv2D

---

## 📌 1. Convolution Operation Kya Hai? (The Math & Intuition)

Deep Learning mein **Convolution** ek mathematical operation hai jisme ek chota sa matrix jise hum **Filter ya Kernel** kehte hain ($3 \times 3$, $5 \times 5$, etc.), woh poori image ke upar left-to-right aur top-to-bottom slide karta hai.

### ⚙️ Kaise Kaam Karta Hai? (Step-by-Step Mechanism):
1. **Overlay**: Filter ko image ke ek chote patch ($3 \times 3$) ke upar rakha jata hai.
2. **Element-wise Multiplication**: Filter ke har weight ko image ke corresponding pixel value se multiply karte hain.
3. **Summation**: Sabhi products ko add karke ek single number (scalar) banate hain.
4. **Bias Addition**: Us scalar mein ek bias term $b$ add karte hain:
   $$z = \sum_{i=1}^{k} \sum_{j=1}^{k} (\text{Image}_{i,j} \times \text{Kernel}_{i,j}) + b$$
5. **Activation**: Us par activation function (jaise ReLU) apply kiya jata hai: $a = \max(0, z)$.
6. **Feature Map**: Yeh calculated value naye matrix (jise **Feature Map** kehte hain) ke corresponding cell mein store ho jaati hai!

---

## 📐 2. Visual Demonstration of Convolution

Agar input image $4 \times 4$ ho aur filter $3 \times 3$ ho:
$$\text{Input: } 4 \times 4 \quad * \quad \text{Filter: } 3 \times 3 \quad \Longrightarrow \quad \text{Output: } (4 - 3 + 1) \times (4 - 3 + 1) = 2 \times 2$$
Aaiye isko pehle **Pure NumPy** mein scratch se banate hain taaki exact logic crystal-clear ho jaye!


In [ ]:
# ============================================================
# STEP 1: Libraries Import
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

print(f"TensorFlow Version: {tf.__version__}")


: 

In [ ]:
# ============================================================
# STEP 2: Custom 2D Convolution from Scratch (Pure NumPy)
# ============================================================

def convolve2d_scratch(image, kernel, bias=0):
    """
    image: 2D numpy array (H, W)
    kernel: 2D numpy array (kH, kW)
    bias: float scalar
    Returns: 2D feature map of shape (H - kH + 1, W - kW + 1)
    """
    H, W = image.shape
    kH, kW = kernel.shape
    
    out_H = H - kH + 1
    out_W = W - kW + 1
    
    # Empty output feature map initialize karte hain
    output = np.zeros((out_H, out_W), dtype=np.float32)
    
    # Sliding window nested loops
    for i in range(out_H):
        for j in range(out_W):
            # Image ka current patch slice karte hain
            patch = image[i:i + kH, j:j + kW]
            # Element-wise multiply + sum + bias
            val = np.sum(patch * kernel) + bias
            output[i, j] = val
            
    return output

# Ek chota sample 4x4 matrix aur 3x3 filter test karte hain
sample_img = np.array([
    [1, 2, 0, 1],
    [0, 1, 3, 2],
    [2, 0, 1, 1],
    [1, 3, 2, 0]
], dtype=np.float32)

sample_kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
], dtype=np.float32)

result_scratch = convolve2d_scratch(sample_img, sample_kernel, bias=0)

print("--- Sample Image (4x4) ---")
print(sample_img)
print("\n--- Sample Vertical Kernel (3x3) ---")
print(sample_kernel)
print("\n--- Convolved Output Feature Map (2x2) ---")
print(result_scratch)


### 💡 Manual Calculation Verification:
Top-Left $3 \times 3$ patch:
```
[1, 2, 0] * [1, 0, -1] = 1*1 + 2*0 + 0*(-1) = 1
[0, 1, 3] * [1, 0, -1] = 0*1 + 1*0 + 3*(-1) = -3
[2, 0, 1] * [1, 0, -1] = 2*1 + 0*0 + 1*(-1) = 1
Sum = 1 + (-3) + 1 = -1
```
Scratch function ka output pehle cell mein exactly **`-1.0`** aaya hai! Math 100% verified!

## 🎨 3. Famous Computer Vision Filters & Unka Effect

Traditional Computer Vision mein log manual filters design karte the:
1. **Sobel Vertical**: Vertical edges (khadi lakeerein) dhoondhta hai.
2. **Sobel Horizontal**: Horizontal edges (leti hui lakeerein) dhoondhta hai.
3. **Sharpen**: Image ke edges ko enhance/sharp karta hai.
4. **Gaussian / Box Blur**: Noise hatane aur image ko smooth karne ke liye.
5. **Laplacian**: Sabhi directions ke edges ek saath detect karta hai.

Aaiye ek real image / handwritten digit par in sabhi filters ko chala kar dekhte hain!

In [ ]:
# ============================================================
# STEP 3: Real Image Filtering with Classic Kernels
# ============================================================

# MNIST se ek digit '8' lete hain jisme vertical, horizontal aur curved edges sab hain
mnist = keras.datasets.mnist
(X_train, y_train), _ = mnist.load_data()
digit_img = X_train[17].astype(np.float32) / 255.0  # Digit image

# Common Filters define karte hain
kernels = {
    "Original": None,
    "Sobel Vertical": np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32),
    
    "Sobel Horizontal": np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype=np.float32),
    
    "Sharpen": np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype=np.float32),
    
    "Box Blur": np.ones((3, 3), dtype=np.float32) / 9.0,
    
    "Laplacian (Edges)": np.array([
        [ 0,  1,  0],
        [ 1, -4,  1],
        [ 0,  1,  0]
    ], dtype=np.float32)
}

# Plot sabhi filtered outputs
plt.figure(figsize=(15, 8))
for idx, (name, k) in enumerate(kernels.items()):
    plt.subplot(2, 3, idx + 1)
    if k is None:
        plt.imshow(digit_img, cmap='gray')
        plt.title(f"{name}\nShape: {digit_img.shape}", fontsize=11, fontweight='bold')
    else:
        filtered = convolve2d_scratch(digit_img, k)
        plt.imshow(filtered, cmap='gray')
        plt.title(f"{name}\nShape: {filtered.shape}", fontsize=11, fontweight='bold')
    plt.axis('off')

plt.suptitle("Custom Filters ka Asar: Feature Maps Visualized", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### 💡 Hinglish Explanation:
Dekhiye:
- **Sobel Vertical** ne digit ke left aur right vertical lines ko highlight kar diya.
- **Sobel Horizontal** ne top aur bottom horizontal lines ko pakad liya.
- **Laplacian** ne poore boundary contour ko nikaal diya.
- **CNN ki khasiyat**: Traditional computer vision mein engineers ko yeh numbers (-1, 0, 1, 2) manually design karne padte the. **Lekin CNN mein model backpropagation ke zariye yeh weights khud-ba-khud learn kar leta hai!**

## 🤖 4. Keras `Conv2D` Layer Implementation

Ab dekhte hain ki TensorFlow/Keras mein `layers.Conv2D` kaise use hoti hai, aur kya Keras wahi same output deta hai jo hamare Scratch code ne diya!

In [ ]:
# ============================================================
# STEP 4: Keras Conv2D Layer vs Scratch Function Verification
# ============================================================

# Sobel Vertical Kernel
sobel_v = kernels["Sobel Vertical"]

# Keras Conv2D layer expect karti hai weights shape: (kH, kW, in_channels, out_channels)
# Yahan 1 input channel aur 1 filter output hai -> shape: (3, 3, 1, 1)
sobel_v_keras = sobel_v.reshape((3, 3, 1, 1))
bias_keras = np.array([0.0], dtype=np.float32)

# Keras Conv2D Layer banate hain (no activation, no padding)
conv_layer = layers.Conv2D(
    filters=1, 
    kernel_size=(3, 3), 
    padding='valid', 
    use_bias=True,
    input_shape=(28, 28, 1)
)

# Dummy call taaki layer build ho jaye
dummy_in = tf.zeros((1, 28, 28, 1))
conv_layer(dummy_in)

# Hamara custom sobel kernel inject karte hain Keras layer ke andar
conv_layer.set_weights([sobel_v_keras, bias_keras])

# Input image ko 4D tensor banate hain: (1, 28, 28, 1)
img_tensor = tf.constant(digit_img.reshape(1, 28, 28, 1))

# Keras se convolution nikaalte hain
keras_output = conv_layer(img_tensor).numpy().squeeze()
scratch_output = convolve2d_scratch(digit_img, sobel_v)

# Check karte hain kya dono identical hain
difference = np.max(np.abs(keras_output - scratch_output))
print(f"✅ Maximum Absolute Difference: {difference:.8f}")
if difference < 1e-5:
    print("🎉 Scratch NumPy aur Keras Conv2D ka result 100% EXACT MATCH hai!")


In [ ]:
# Visual Verification
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(scratch_output, cmap='gray')
ax[0].set_title(f"Custom Scratch Output\nShape: {scratch_output.shape}", fontweight='bold')
ax[0].axis('off')

ax[1].imshow(keras_output, cmap='gray')
ax[1].set_title(f"Keras Conv2D Output\nShape: {keras_output.shape}", fontweight='bold')
ax[1].axis('off')

plt.suptitle("Scratch vs Keras: 100% Identical Output", fontsize=13, fontweight='bold')
plt.show()


## 📝 Summary & Key Takeaways in Hinglish

1. **Convolution is Feature Extraction**: Filter image par ghoomta hai aur matching patterns (edges, lines) dhoondhkar highlight karta hai.
2. **Trainable Filters**: Keras `Conv2D` random weights se shuru hota hai aur training ke waqt khud best filters learn kar leta hai.
3. **Valid Output Size**: Agar $N \times N$ image par $F \times F$ filter lagayein without padding, toh output size hamesha $(N - F + 1)$ hota hai (image shuru mein shrink hoti hai).
